PANDAS 

In [32]:
!pip install pandas numpy

1244.14s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [33]:
!pip install scikit-learn

1249.72s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


LIBRERIAS 


In [34]:
import pandas as pd
import numpy as np
import time

In [35]:
ruta_archivo = '/home/paula/MCEI_2620/Datos_MCEI.csv'

In [36]:
df = pd.read_csv(ruta_archivo, sep=';')

print("¡Archivo cargado exitosamente (separado por ';')!")
print("\nPrimeras filas del dataset:")
display(df.head())

print("\nInformación general del dataset:")
print(df.info())

¡Archivo cargado exitosamente (separado por ';')!

Primeras filas del dataset:


,muestra,voltaje_V,corriente_A,potencia_W_aprox
0,1,12021,475,56562
1,2,11918,415,49541
2,3,12195,60,73315
3,4,12083,386,45804
4,5,11979,588,70091



Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   muestra           100 non-null    int64
 1   voltaje_V         100 non-null    int64
 2   corriente_A       100 non-null    int64
 3   potencia_W_aprox  100 non-null    int64
dtypes: int64(4)
memory usage: 3.3 KB
None


In [37]:
from sklearn.neighbors import NearestNeighbors

In [38]:
# %%
# Extraemos las características usando los nombres exactos de tus columnas
X = df[['voltaje_V', 'corriente_A', 'potencia_W_aprox']].values

# Identificamos el sensor con la anomalía (el que presenta el voltaje más alto)
idx_anomalia = df['voltaje_V'].idxmax()
punto_anomalo = X[idx_anomalia].reshape(1, -1)

print(f"=== Sensor Anómalo Detectado (Índice {idx_anomalia}) ===")
print(df.loc[idx_anomalia])
print("-" * 50)

# Aplicamos KNN Estándar (k=5 vecinos más cercanos, usando n_neighbors=6 para excluir el punto mismo)
from sklearn.neighbors import NearestNeighbors
knn = NearestNeighbors(n_neighbors=6, algorithm='auto', metric='euclidean')
knn.fit(X)
distancias_knn, indices_knn = knn.kneighbors(punto_anomalo)

print("--- Resultados con KNN Estándar ---")
for i in range(1, 6):
    idx_v = indices_knn[0][i]
    print(f"Vecino {i} | Índice: {idx_v} | Distancia: {distancias_knn[0][i]:.4f}")

=== Sensor Anómalo Detectado (Índice 81) ===
muestra                82
voltaje_V           12447
corriente_A           537
potencia_W_aprox    66262
Name: 81, dtype: int64
--------------------------------------------------
--- Resultados con KNN Estándar ---
Vecino 1 | Índice: 50 | Distancia: 344.9188
Vecino 2 | Índice: 68 | Distancia: 382.0798
Vecino 3 | Índice: 76 | Distancia: 476.9612
Vecino 4 | Índice: 29 | Distancia: 749.0387
Vecino 5 | Índice: 14 | Distancia: 838.9029


In [39]:
kdtree = NearestNeighbors(n_neighbors=6, algorithm='kd_tree', metric='euclidean')
kdtree.fit(X)
distancias_kd, indices_kd = kdtree.kneighbors(punto_anomalo)

print("--- Resultados usando kd-tree ---")
for i in range(1, 6):
    idx_v = indices_kd[0][i]
    print(f"Vecino {i} | Índice: {idx_v} | Distancia: {distancias_kd[0][i]:.4f}")

--- Resultados usando kd-tree ---
Vecino 1 | Índice: 50 | Distancia: 344.9188
Vecino 2 | Índice: 68 | Distancia: 382.0798
Vecino 3 | Índice: 76 | Distancia: 476.9612
Vecino 4 | Índice: 29 | Distancia: 749.0387
Vecino 5 | Índice: 14 | Distancia: 838.9029


In [40]:
t0 = time.time()
knn_bench = NearestNeighbors(n_neighbors=6, algorithm='auto', metric='euclidean').fit(X)
_, _ = knn_bench.kneighbors(punto_anomalo)
tiempo_knn = time.time() - t0

# Medir tiempo con la estructura kd-tree
t0 = time.time()
kd_bench = NearestNeighbors(n_neighbors=6, algorithm='kd_tree', metric='euclidean').fit(X)
_, _ = kd_bench.kneighbors(punto_anomalo)
tiempo_kd = time.time() - t0

print("--- Comparación de Tiempos de Ejecución ---")
print(f"Tiempo KNN (automático/fuerza bruta): {tiempo_knn:.6f} segundos")
print(f"Tiempo kd-tree:                       {tiempo_kd:.6f} segundos")

--- Comparación de Tiempos de Ejecución ---
Tiempo KNN (automático/fuerza bruta): 0.003681 segundos
Tiempo kd-tree:                       0.002890 segundos
